<a href="https://colab.research.google.com/github/Kishoby/Churn_Prediction_ML/blob/Feature-Engineering/Churn_Feature_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import os

In [5]:
df = pd.read_csv(
    "/content/drive/MyDrive/Project_Churn Prediction/Telco_Customer_Churn_Cleaned.csv"
)

print("Dataset Loaded Successfully")
print("Dataset Shape:", df.shape)

display(df.head())

Dataset Loaded Successfully
Dataset Shape: (7032, 31)


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,0,False,True,False,False,True,...,False,False,False,False,False,False,True,False,True,False
1,0,34,56.95,1889.50,0,True,False,False,True,False,...,False,False,False,False,True,False,False,False,False,True
2,0,2,53.85,108.15,1,True,False,False,True,False,...,False,False,False,False,False,False,True,False,False,True
3,0,45,42.30,1840.75,0,True,False,False,False,True,...,False,False,False,False,True,False,False,False,False,False
4,0,2,70.70,151.65,1,False,False,False,True,False,...,False,False,False,False,False,False,True,False,True,False


In [6]:
feature_path = "/content/drive/MyDrive/Project_Churn Prediction/02_Feature_Engineering"

os.makedirs(feature_path, exist_ok=True)

print("Feature Engineering folder created successfully.")

Feature Engineering folder created successfully.


In [7]:
required_columns = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "Churn"
]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Required columns are missing: {missing_columns}"
    )

print("All required columns are available.")

All required columns are available.


In [8]:
feature_df = df.copy()

print("Original dataset shape:", df.shape)
print("Working dataset shape:", feature_df.shape)

Original dataset shape: (7032, 31)
Working dataset shape: (7032, 31)


In [9]:
feature_df["AverageMonthlySpend"] = np.where(
    feature_df["tenure"] > 0,
    feature_df["TotalCharges"] / feature_df["tenure"],
    feature_df["MonthlyCharges"]
)

In [10]:
feature_df["TenureGroup"] = pd.cut(
    feature_df["tenure"],
    bins=[-1, 12, 24, 48, 60, np.inf],
    labels=[
        "0-12 Months",
        "13-24 Months",
        "25-48 Months",
        "49-60 Months",
        "Above 60 Months"
    ]
)

display(
    feature_df["TenureGroup"].value_counts()
)

,count
TenureGroup,
0-12 Months,2175
25-48 Months,1594
Above 60 Months,1407
13-24 Months,1024
49-60 Months,832


In [11]:
tenure_dummies = pd.get_dummies(
    feature_df["TenureGroup"],
    prefix="TenureGroup",
    dtype=int
)

feature_df = pd.concat(
    [feature_df, tenure_dummies],
    axis=1
)

feature_df.drop(
    columns=["TenureGroup"],
    inplace=True
)

In [12]:
feature_df["NewCustomer"] = (
    feature_df["tenure"] <= 12
).astype(int)

feature_df["LongTermCustomer"] = (
    feature_df["tenure"] >= 48
).astype(int)

In [13]:
feature_df["LowMonthlyCharges"] = (
    feature_df["MonthlyCharges"] < 35
).astype(int)

feature_df["MediumMonthlyCharges"] = (
    feature_df["MonthlyCharges"].between(
        35,
        70,
        inclusive="left"
    )
).astype(int)

feature_df["HighMonthlyCharges"] = (
    feature_df["MonthlyCharges"] >= 70
).astype(int)

In [14]:
feature_df["LowTotalCharges"] = (
    feature_df["TotalCharges"] < 500
).astype(int)

feature_df["HighTotalCharges"] = (
    feature_df["TotalCharges"] >= 3000
).astype(int)

In [15]:
contract_one_year = feature_df.get(
    "Contract_One year",
    pd.Series(0, index=feature_df.index)
).astype(int)

contract_two_year = feature_df.get(
    "Contract_Two year",
    pd.Series(0, index=feature_df.index)
).astype(int)

feature_df["ContractDurationMonths"] = (
    1
    + 11 * contract_one_year
    + 23 * contract_two_year
)

In [16]:
feature_df["MonthToMonthContract"] = (
    (contract_one_year == 0) &
    (contract_two_year == 0)
).astype(int)

feature_df["LongTermContract"] = (
    (contract_one_year == 1) |
    (contract_two_year == 1)
).astype(int)

In [17]:
credit_card = feature_df.get(
    "PaymentMethod_Credit card (automatic)",
    pd.Series(0, index=feature_df.index)
).astype(int)

bank_transfer = feature_df.get(
    "PaymentMethod_Bank transfer (automatic)",
    pd.Series(0, index=feature_df.index)
).astype(int)

feature_df["AutomaticPayment"] = (
    (credit_card == 1) |
    (bank_transfer == 1)
).astype(int)

In [20]:
streaming_columns = [
    "StreamingTV_Yes",
    "StreamingMovies_Yes"
]

existing_streaming_columns = [
    column for column in streaming_columns
    if column in feature_df.columns
]

feature_df["StreamingServiceCount"] = (
    feature_df[existing_streaming_columns]
    .astype(int)
    .sum(axis=1)
)

In [21]:
additional_service_columns = (
    existing_support_columns
    + existing_streaming_columns
)

if "MultipleLines_Yes" in feature_df.columns:
    additional_service_columns.append(
        "MultipleLines_Yes"
    )

feature_df["TotalAdditionalServices"] = (
    feature_df[additional_service_columns]
    .astype(int)
    .sum(axis=1)
)

print("Additional service columns used:")
print(additional_service_columns)

Additional service columns used:
['OnlineSecurity_Yes', 'OnlineBackup_Yes', 'DeviceProtection_Yes', 'TechSupport_Yes', 'StreamingTV_Yes', 'StreamingMovies_Yes', 'MultipleLines_Yes']


In [22]:
online_security = feature_df.get(
    "OnlineSecurity_Yes",
    pd.Series(0, index=feature_df.index)
).astype(int)

tech_support = feature_df.get(
    "TechSupport_Yes",
    pd.Series(0, index=feature_df.index)
).astype(int)

feature_df["NoSecurityOrTechSupport"] = (
    (online_security == 0) &
    (tech_support == 0)
).astype(int)

In [23]:
no_internet = feature_df.get(
    "InternetService_No",
    pd.Series(0, index=feature_df.index)
).astype(int)

fiber_optic = feature_df.get(
    "InternetService_Fiber optic",
    pd.Series(0, index=feature_df.index)
).astype(int)

feature_df["HasInternetService"] = (
    no_internet == 0
).astype(int)

feature_df["FiberOpticCustomer"] = fiber_optic

In [24]:
electronic_check = feature_df.get(
    "PaymentMethod_Electronic check",
    pd.Series(0, index=feature_df.index)
).astype(int)

feature_df["HighRiskBillingProfile"] = (
    (feature_df["HighMonthlyCharges"] == 1) &
    (feature_df["MonthToMonthContract"] == 1) &
    (electronic_check == 1)
).astype(int)

In [25]:
feature_df["ChargesPerContractMonth"] = (
    feature_df["MonthlyCharges"]
    / feature_df["ContractDurationMonths"]
)

In [26]:
feature_df["Tenure_MonthlyCharges_Interaction"] = (
    feature_df["tenure"]
    * feature_df["MonthlyCharges"]
)

feature_df["Tenure_ServiceCount_Interaction"] = (
    feature_df["tenure"]
    * feature_df["TotalAdditionalServices"]
)

feature_df["MonthlyCharges_ServiceCount_Interaction"] = (
    feature_df["MonthlyCharges"]
    * feature_df["TotalAdditionalServices"]
)

In [27]:
original_columns = set(df.columns)
new_columns = [
    column for column in feature_df.columns
    if column not in original_columns
]

print("Number of original features:", len(df.columns))
print("Number of new features:", len(new_columns))
print("Final number of columns:", len(feature_df.columns))

print("\nNewly created features:")

for column in new_columns:
    print(column)

Number of original features: 31
Number of new features: 28
Final number of columns: 59

Newly created features:
AverageMonthlySpend
TenureGroup_0-12 Months
TenureGroup_13-24 Months
TenureGroup_25-48 Months
TenureGroup_49-60 Months
TenureGroup_Above 60 Months
NewCustomer
LongTermCustomer
LowMonthlyCharges
MediumMonthlyCharges
HighMonthlyCharges
LowTotalCharges
HighTotalCharges
ContractDurationMonths
MonthToMonthContract
LongTermContract
AutomaticPayment
SupportServiceCount
StreamingServiceCount
TotalAdditionalServices
NoSecurityOrTechSupport
HasInternetService
FiberOpticCustomer
HighRiskBillingProfile
ChargesPerContractMonth
Tenure_MonthlyCharges_Interaction
Tenure_ServiceCount_Interaction
MonthlyCharges_ServiceCount_Interaction


In [28]:
print("Final dataset shape:", feature_df.shape)

display(feature_df.head())
feature_df.info()missing_values = feature_df.isnull().sum().sum()

numeric_df = feature_df.select_dtypes(
    include=[np.number]
)

infinite_values = np.isinf(
    numeric_df
).sum().sum()

print("Total missing values:", missing_values)
print("Total infinite values:", infinite_values)

Final dataset shape: (7032, 59)


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,...,StreamingServiceCount,TotalAdditionalServices,NoSecurityOrTechSupport,HasInternetService,FiberOpticCustomer,HighRiskBillingProfile,ChargesPerContractMonth,Tenure_MonthlyCharges_Interaction,Tenure_ServiceCount_Interaction,MonthlyCharges_ServiceCount_Interaction
0,0,1,29.85,29.85,0,False,True,False,False,True,...,0,1,1,1,0,0,29.850000,29.85,1,29.85
1,0,34,56.95,1889.50,0,True,False,False,True,False,...,0,2,0,1,0,0,4.745833,1936.30,68,113.90
2,0,2,53.85,108.15,1,True,False,False,True,False,...,0,2,0,1,0,0,53.850000,107.70,4,107.70
3,0,45,42.30,1840.75,0,True,False,False,False,True,...,0,3,0,1,0,0,3.525000,1903.50,135,126.90
4,0,2,70.70,151.65,1,False,False,False,True,False,...,0,0,1,1,1,1,70.700000,141.40,0,0.00


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7032 entries, 0 to 7031
Data columns (total 59 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   SeniorCitizen                            7032 non-null   int64  
 1   tenure                                   7032 non-null   int64  
 2   MonthlyCharges                           7032 non-null   float64
 3   TotalCharges                             7032 non-null   float64
 4   Churn                                    7032 non-null   int64  
 5   gender_Male                              7032 non-null   bool   
 6   Partner_Yes                              7032 non-null   bool   
 7   Dependents_Yes                           7032 non-null   bool   
 8   PhoneService_Yes                         7032 non-null   bool   
 9   MultipleLines_No phone service           7032 non-null   bool   
 10  MultipleLines_Yes                        7032 no

In [29]:
missing_values = feature_df.isnull().sum().sum()

numeric_df = feature_df.select_dtypes(
    include=[np.number]
)

infinite_values = np.isinf(
    numeric_df
).sum().sum()

print("Total missing values:", missing_values)
print("Total infinite values:", infinite_values)

Total missing values: 0
Total infinite values: 0


In [30]:
feature_df.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True
)

if feature_df.isnull().sum().sum() > 0:
    feature_df = feature_df.fillna(0)

print(
    "Missing values after validation:",
    feature_df.isnull().sum().sum()
)

Missing values after validation: 0


In [31]:
feature_summary = pd.DataFrame({
    "Feature": new_columns,
    "Data_Type": [
        str(feature_df[column].dtype)
        for column in new_columns
    ],
    "Minimum": [
        feature_df[column].min()
        for column in new_columns
    ],
    "Maximum": [
        feature_df[column].max()
        for column in new_columns
    ],
    "Mean": [
        feature_df[column].mean()
        for column in new_columns
    ]
})

display(feature_summary)

,Feature,Data_Type,Minimum,Maximum,Mean
0,AverageMonthlySpend,float64,13.775000,121.40,64.799424
1,TenureGroup_0-12 Months,int64,0.000000,1.00,0.309300
2,TenureGroup_13-24 Months,int64,0.000000,1.00,0.145620
3,TenureGroup_25-48 Months,int64,0.000000,1.00,0.226678
4,TenureGroup_49-60 Months,int64,0.000000,1.00,0.118316
5,TenureGroup_Above 60 Months,int64,0.000000,1.00,0.200085
6,NewCustomer,int64,0.000000,1.00,0.309300
7,LongTermCustomer,int64,0.000000,1.00,0.327503
8,LowMonthlyCharges,int64,0.000000,1.00,0.245307
9,MediumMonthlyCharges,int64,0.000000,1.00,0.244312


In [32]:
feature_output_file = (
    f"{feature_path}/"
    "Telco_Customer_Churn_Feature_Engineered.csv"
)

feature_df.to_csv(
    feature_output_file,
    index=False
)

print("Feature-engineered dataset saved successfully:")
print(feature_output_file)

Feature-engineered dataset saved successfully:
/content/drive/MyDrive/Project_Churn Prediction/02_Feature_Engineering/Telco_Customer_Churn_Feature_Engineered.csv


In [33]:
feature_summary_file = (
    f"{feature_path}/"
    "Feature_Engineering_Summary.csv"
)

feature_summary.to_csv(
    feature_summary_file,
    index=False
)

print("Feature summary saved successfully:")
print(feature_summary_file)

Feature summary saved successfully:
/content/drive/MyDrive/Project_Churn Prediction/02_Feature_Engineering/Feature_Engineering_Summary.csv


In [34]:
feature_descriptions = {
    "AverageMonthlySpend":
        "Average total charges paid per month of tenure.",

    "NewCustomer":
        "Customer with tenure of 12 months or less.",

    "LongTermCustomer":
        "Customer with tenure of at least 48 months.",

    "LowMonthlyCharges":
        "Customer paying less than 35 per month.",

    "MediumMonthlyCharges":
        "Customer paying between 35 and below 70 per month.",

    "HighMonthlyCharges":
        "Customer paying 70 or more per month.",

    "LowTotalCharges":
        "Customer with total charges below 500.",

    "HighTotalCharges":
        "Customer with total charges of at least 3000.",

    "ContractDurationMonths":
        "Numerical representation of contract duration.",

    "MonthToMonthContract":
        "Customer using a month-to-month contract.",

    "LongTermContract":
        "Customer using a one-year or two-year contract.",

    "AutomaticPayment":
        "Customer using an automatic payment method.",

    "SupportServiceCount":
        "Number of support and protection services used.",

    "StreamingServiceCount":
        "Number of streaming services used.",

    "TotalAdditionalServices":
        "Total number of selected additional services.",

    "NoSecurityOrTechSupport":
        "Customer without online security and technical support.",

    "HasInternetService":
        "Customer currently using an internet service.",

    "FiberOpticCustomer":
        "Customer using fibre-optic internet.",

    "HighRiskBillingProfile":
        "High-charge, month-to-month, electronic-check profile.",

    "ChargesPerContractMonth":
        "Monthly charges divided by contract duration.",

    "Tenure_MonthlyCharges_Interaction":
        "Interaction between tenure and monthly charges.",

    "Tenure_ServiceCount_Interaction":
        "Interaction between tenure and additional service count.",

    "MonthlyCharges_ServiceCount_Interaction":
        "Interaction between monthly charges and service count."
}

feature_dictionary = pd.DataFrame(
    feature_descriptions.items(),
    columns=["Feature", "Description"]
)

feature_dictionary.to_csv(
    f"{feature_path}/Feature_Dictionary.csv",
    index=False
)

display(feature_dictionary)

,Feature,Description
0,AverageMonthlySpend,Average total charges paid per month of tenure.
1,NewCustomer,Customer with tenure of 12 months or less.
2,LongTermCustomer,Customer with tenure of at least 48 months.
3,LowMonthlyCharges,Customer paying less than 35 per month.
4,MediumMonthlyCharges,Customer paying between 35 and below 70 per mo...
5,HighMonthlyCharges,Customer paying 70 or more per month.
6,LowTotalCharges,Customer with total charges below 500.
7,HighTotalCharges,Customer with total charges of at least 3000.
8,ContractDurationMonths,Numerical representation of contract duration.
9,MonthToMonthContract,Customer using a month-to-month contract.


In [35]:
feature_df.to_csv(
    f"{feature_path}/Telco_Customer_Churn_Feature_Engineered.csv",
    index=False
)

print("Feature engineered dataset saved.")

Feature engineered dataset saved.


In [36]:
feature_summary.to_csv(
    f"{feature_path}/Feature_Engineering_Summary.csv",
    index=False
)

print("Feature engineering summary saved.")

Feature engineering summary saved.


In [37]:
feature_dictionary.to_csv(
    f"{feature_path}/Feature_Dictionary.csv",
    index=False
)

print("Feature dictionary saved.")

Feature dictionary saved.


In [38]:
import shutil

shutil.make_archive(
    "/content/Feature_Engineering",
    "zip",
    feature_path
)

print("ZIP file created successfully.")

ZIP file created successfully.


In [39]:
from google.colab import files

files.download("/content/Feature_Engineering.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>